In [1]:
import pandas as pd
from icsa import ICSA, set_seeds
import plotly.express as px
from network_analysis import load_network_results
import kaleido
from plotly.subplots import make_subplots

set_seeds(0)

In [2]:
# KNN #
knn_fpath = 'analysis/classification/knn/'  # filepath for analysis files
results_path_overlapping_knn = knn_fpath + 'mcc/overlapping/'
results, completed_idxs, results_df = load_network_results(results_path_overlapping_knn)
results_df = pd.DataFrame(results_df)

In [3]:
results_path_concatenated_knn = knn_fpath + 'mcc/concatenated/'
results_concat, completed_idxs_concat, results_df_concat = load_network_results(results_path_concatenated_knn)
results_df_concat = pd.DataFrame(results_df_concat)

In [4]:
results_df = results_df.rename(columns={'instance_encoder_num_hidden_layers': 'IE_NHL',
                                        'instance_decoder_num_hidden_layers': 'ID_NHL',
                                        'configuration_encoder_num_hidden_layers': 'CE_NHL'})

In [5]:
results_df_concat = results_df_concat.rename(columns={'instance_encoder_num_hidden_layers': 'IE_NHL',
                                                      'instance_decoder_num_hidden_layers': 'ID_NHL',
                                                      'configuration_encoder_num_hidden_layers': 'CE_NHL',
                                                      'performance_num_hidden_layers': 'Performance_NHL'})

In [6]:
# results_df = results_df.sort_values(by='CE_NHL')

In [7]:
results_df['num_hidden_units'] = results_df['num_hidden_units'].astype('string').astype('category')
results_df_concat['num_hidden_units'] = results_df_concat['num_hidden_units'].astype('string').astype('category')

In [8]:
category_orders = {
    'IE_NHL': [0, 1, 2, 3],
    'ID_NHL': [0, 1, 2, 3],
    'CE_NHL': [0, 1, 2, 3],
    'Performance_NHL': [0, 1, 2, 3],
    'num_hidden_units': [16, 32, 64],
    'activation': ['relu', 'leaky_relu', 'tanh']
}

In [9]:
last_epoch = results_df[(results_df['epoch'] == 10)].index
last_epoch_concat = results_df_concat[(results_df_concat['epoch'] == 10)].index

In [10]:
def layout_update_save(fig, name: str, mode: str, save: bool = True, width: int = 1000, height: int = 1000):
    fig = fig.update_layout(
        # margin = {'t': 0, 'b': 0, 'l': 0, 'r': 0}, 
        font = dict(size=20),
        width=width, height=height, template = 'plotly_white'
    )
    if save:
        fig.write_image(knn_fpath + 'mcc/' + name + '_' + mode + '.png', engine='kaleido')
    return fig

def subplotter(df: pd.DataFrame, x, color=None, facet_col=None, facet_row=None):
    x_axis_title = x
    fig = make_subplots(rows=2, cols=1, vertical_spacing=0.02, shared_xaxes=True)
    fig.add_trace(px.box(df, x=x, y = 'val_performance_loss', color=color, facet_col=facet_col, facet_row=facet_row, category_orders=category_orders).data[0], row=1, col=1)
    fig.update_yaxes(title_text='val_performance_loss', row=1, col=1)
    fig.add_trace(px.box(df, x=x, y = 'val_feature_loss', color=color, facet_col=facet_col, facet_row=facet_row, category_orders=category_orders).data[0], row=2, col=1)
    fig.update_yaxes(title_text='val_feature_loss', row=2, col=1)
    fig.update_xaxes(title_text=x_axis_title, row=2, col=1)
    return fig
    
    

In [11]:
fig = subplotter(results_df, x='epoch')

In [ ]:
layout_update_save(fig, 'val_loss_vs_epoch', 'overlaying')

In [13]:
fig_concat = subplotter(results_df_concat, x='epoch')

In [ ]:
layout_update_save(fig_concat, 'val_loss_vs_epoch', 'concat')

In [15]:
results_df2 = results_df.copy()
results_df2['model_type'] = 'overlaying'

results_df_concat2 = results_df_concat.copy()
results_df_concat2['model_type'] = 'concatenate'

results_df_both2 = pd.concat([results_df2, results_df_concat2], ignore_index=True)

In [16]:
last_epoch2 = results_df_both2[(results_df_both2['epoch'] == 10)].index

In [ ]:
fig = px.box(results_df_both2, x='epoch', y = 'val_performance_loss', color='model_type', category_orders=category_orders)
layout_update_save(fig, 'val_loss_vs_epoch_performance', 'both')

In [ ]:
fig = px.box(results_df_both2, x='epoch', y = 'val_feature_loss', color='model_type', category_orders=category_orders)
layout_update_save(fig, 'val_loss_vs_epoch_feature', 'both')

In [19]:
fig = px.box(results_df.loc[last_epoch], x = 'CE_NHL', y = 'val_performance_loss', color='ID_NHL', facet_col='IE_NHL',
             category_orders=category_orders, color_discrete_sequence=px.colors.sequential.Blues_r)

In [ ]:
layout_update_save(fig, 'network_architecture_experiments_1_performance_loss', 'overlaying', save=True, height=500)

In [ ]:
fig = px.box(results_df.loc[last_epoch], x = 'CE_NHL', y = 'val_feature_loss', color='ID_NHL', facet_col='IE_NHL',
             category_orders=category_orders, color_discrete_sequence=px.colors.sequential.Blues_r)
layout_update_save(fig, 'network_architecture_experiments_1_feature_loss', 'overlaying', save=True, height=500)

In [ ]:
fig = px.box(results_df.loc[last_epoch], x = 'num_hidden_units', y = 'val_performance_loss', color='activation',
             category_orders=category_orders)
layout_update_save(fig, 'network_architecture_experiments_2_performance_loss', 'overlaying', save=True)

In [ ]:
fig = px.box(results_df.loc[last_epoch], x = 'num_hidden_units', y = 'val_feature_loss', color='activation',
             category_orders=category_orders)
layout_update_save(fig, 'network_architecture_experiments_2_feature_loss', 'overlaying', save=True)

In [ ]:
fig = px.box(results_df_both2.loc[last_epoch2], x = 'IE_NHL', y = 'val_performance_loss',
             facet_col = 'model_type', color='ID_NHL', category_orders=category_orders, color_discrete_sequence=px.colors.sequential.Blues_r)
layout_update_save(fig, 'network_architecture_experiments_3_performance_loss', 'both', save=True)

In [ ]:
fig = px.box(results_df_both2.loc[last_epoch2], x = 'IE_NHL', y = 'val_feature_loss',
             facet_col = 'model_type', color='ID_NHL', category_orders=category_orders, color_discrete_sequence=px.colors.sequential.Blues_r)
layout_update_save(fig, 'network_architecture_experiments_3_feature_loss', 'both', save=True)